# Chapter 11: Probabilistic Graphical Models

```{admonition} Learning Objectives
:class: tip
- Understand Bayesian networks and their semantics
- Master inference algorithms (Variable Elimination, Belief Propagation)
- Apply Hidden Markov Models (HMMs)
- Implement Forward-Backward and Viterbi algorithms
- Learn parameters with EM algorithm
- Understand Markov Random Fields
- Build probabilistic reasoning systems
```

```{epigraph}
Probabilistic graphical models combine graph theory and probability theory to provide a natural framework for representing and reasoning about uncertainty.

-- Daphne Koller & Nir Friedman
```

## 11.1 Introduction

**Probabilistic Graphical Models (PGMs)** represent complex probability distributions using graphs.

### Motivation

**Challenge**: High-dimensional probability distributions
- $n$ binary variables: $2^n$ parameters
- 30 variables: $>1$ billion parameters
- Impossible to store or learn

**Solution**: Exploit independence structure
- Graph encodes conditional independence
- Factorize joint distribution
- Exponentially fewer parameters

### Types of Graphical Models

| Model | Graph Type | Edges | Semantics |
|-------|-----------|-------|-----------|
| Bayesian Network | Directed (DAG) | Causal | $P(X_i \mid \text{Parents}(X_i))$ |
| Markov Network | Undirected | Correlation | $\phi(X_C)$ potentials |
| Hidden Markov Model | Directed | Temporal | Sequential observations |

### Applications

- **Medical diagnosis**: Disease → Symptoms
- **Speech recognition**: Acoustic → Phonemes → Words
- **Computer vision**: Image → Objects → Scene
- **Robotics**: Sensor fusion, SLAM
- **Natural language**: Part-of-speech tagging, parsing
- **Bioinformatics**: Gene regulatory networks

## 11.2 Bayesian Networks

A **Bayesian Network** is a directed acyclic graph (DAG) representing a factorized probability distribution.

### 11.2.1 Definition

**Components**:
1. **Nodes**: Random variables $X_1, ..., X_n$
2. **Edges**: $X_i \to X_j$ means $X_i$ directly influences $X_j$
3. **CPDs**: Conditional probability distributions $P(X_i | \text{Parents}(X_i))$

**Joint distribution** factorizes:
$$P(X_1, ..., X_n) = \prod_{i=1}^{n} P(X_i | \text{Parents}(X_i))$$

### 11.2.2 Example: Medical Diagnosis

```
    Smoking
       |
       v
    Cancer
     /  \
    v    v
 X-ray  Dyspnea
```

**Variables**:
- $S$: Smoking (yes/no)
- $C$: Cancer (yes/no)
- $X$: X-ray positive (yes/no)
- $D$: Dyspnea (yes/no)

**Joint distribution**:
$$P(S,C,X,D) = P(S) \cdot P(C|S) \cdot P(X|C) \cdot P(D|C)$$

**Parameters**: $1 + 2 + 2 + 2 = 7$ vs $2^4-1 = 15$ for full joint

### 11.2.3 Conditional Independence

**Key property**: Graph structure encodes independence

**D-separation**: $X$ and $Y$ are conditionally independent given $Z$ if all paths between $X$ and $Y$ are blocked by $Z$

**Three canonical patterns**:

**1. Chain**: $X \to Z \to Y$
- $X \perp Y | Z$ (Z blocks)

**2. Fork**: $X \leftarrow Z \to Y$
- $X \perp Y | Z$ (Z blocks)

**3. V-structure (Collider)**: $X \to Z \leftarrow Y$
- $X \perp Y$ (Z doesn't block unless observed)
- $X \not\perp Y | Z$ (Explaining away effect)

### 11.2.4 Constructing Bayesian Networks

**Procedure**:
1. Choose ordering of variables
2. For each variable $X_i$:
   - Add edges from minimal set of predecessors that renders $X_i$ independent of others
   - Specify $P(X_i | \text{Parents}(X_i))$

**Result**: Graph is DAG by construction

## 11.3 Inference in Bayesian Networks

**Inference task**: Compute posterior $P(X|E)$ given evidence $E$

### 11.3.1 Inference Types

**1. Marginal inference**: $P(X)$
**2. Conditional inference**: $P(X|E)$
**3. MAP inference**: $\arg\max_x P(X=x|E)$

### 11.3.2 Naive Enumeration

**Marginal inference** by summing out variables:

$$P(X) = \sum_{Y_1,...,Y_m} P(X, Y_1, ..., Y_m)$$

**Problem**: Exponential in number of variables

**Example**: Compute $P(C)$ in medical network
$$P(C) = \sum_{S,X,D} P(S,C,X,D)$$

Requires $2^3 = 8$ terms

### 11.3.3 Variable Elimination

**Key insight**: Push summations inside products

**Example**:
$$\begin{align}
P(C) &= \sum_S \sum_X \sum_D P(S)P(C|S)P(X|C)P(D|C) \\
&= \sum_S P(S)P(C|S) \sum_X P(X|C) \sum_D P(D|C) \\
&= \sum_S P(S)P(C|S) \cdot 1 \cdot 1
\end{align}$$

**Complexity**: Linear in number of variables (for this example)

### 11.3.4 Variable Elimination Algorithm

```
Algorithm: VARIABLE-ELIMINATION(BN, query X, evidence E)

begin
    // Initialize factors from CPDs
    factors ← {P(X_i | Parents(X_i)) for all variables}
    
    // Incorporate evidence
    for each evidence variable E_j = e_j do
        Reduce factors by setting E_j = e_j
    
    // Determine elimination order
    order ← Variables \ (X ∪ E)
    
    // Eliminate variables one by one
    for each Y in order do
        // Collect factors mentioning Y
        relevant ← {f ∈ factors : Y ∈ scope(f)}
        
        // Multiply factors
        product ← ∏_{f ∈ relevant} f
        
        // Sum out Y
        new_factor ← ∑_Y product
        
        // Update factors
        factors ← (factors \ relevant) ∪ {new_factor}
    
    // Multiply remaining factors
    result ← ∏_{f ∈ factors} f
    
    // Normalize
    return result / ∑_X result
end
```

### 11.3.5 Complexity

**Time/Space**: Exponential in **treewidth** of graph
- Treewidth: Size of largest factor during elimination
- Depends on elimination order
- Finding optimal order is NP-hard

**Heuristics**:
- Min-degree: Eliminate variable with fewest neighbors
- Min-fill: Minimize edges added to graph

**For trees**: Linear time (treewidth = 1)

## 11.4 Belief Propagation

**Belief Propagation** (Sum-Product algorithm) efficiently computes marginals on trees.

### 11.4.1 Message Passing

**Idea**: Nodes send messages to neighbors

**Message** from $X_i$ to $X_j$:
$$m_{i\to j}(x_j) = \sum_{x_i} \psi(x_i, x_j) \phi(x_i) \prod_{k \in N(i) \setminus j} m_{k\to i}(x_i)$$

where:
- $\psi(x_i, x_j)$: Edge potential (compatibility)
- $\phi(x_i)$: Node potential (local evidence)
- $N(i)$: Neighbors of node $i$

**Marginal** at node $X_i$:
$$P(x_i) \propto \phi(x_i) \prod_{k \in N(i)} m_{k\to i}(x_i)$$

### 11.4.2 Algorithm for Trees

```
Algorithm: BELIEF-PROPAGATION(Tree T, evidence E)

begin
    // Initialize messages
    for each edge (i,j) do
        m_{i→j}(x_j) ← uniform
    
    // Incorporate evidence
    for each evidence E_i = e_i do
        φ(x_i) ← 1 if x_i = e_i, else 0
    
    // Forward pass (leaves to root)
    Choose arbitrary root r
    for each node i in post-order do
        for each neighbor j of i (toward root) do
            m_{i→j}(x_j) ← ∑_{x_i} ψ(x_i,x_j) φ(x_i) ∏_{k∈N(i)\j} m_{k→i}(x_i)
    
    // Backward pass (root to leaves)
    for each node i in pre-order do
        for each neighbor j of i (away from root) do
            m_{i→j}(x_j) ← ∑_{x_i} ψ(x_i,x_j) φ(x_i) ∏_{k∈N(i)\j} m_{k→i}(x_i)
    
    // Compute marginals
    for each node i do
        P(x_i) ← normalize(φ(x_i) ∏_{k∈N(i)} m_{k→i}(x_i))
    
    return P
end
```

### 11.4.3 Properties

**For trees**:
- **Exact**: Computes exact marginals
- **Efficient**: $O(n \cdot k^2)$ for $n$ nodes, $k$ states
- **Convergence**: Two passes sufficient

**For graphs with cycles** (Loopy BP):
- **Approximate**: May not converge or give wrong answer
- **Often works well** in practice
- Iterate messages until convergence

## 11.5 Hidden Markov Models (HMMs)

**HMM** is a temporal probabilistic model for sequence data.

### 11.5.1 HMM Definition

**Components**:
1. **Hidden states**: $S_t \in \{s_1, ..., s_N\}$ (not observed)
2. **Observations**: $O_t \in \{o_1, ..., o_M\}$ (observed)
3. **Transition probabilities**: $A_{ij} = P(S_{t+1}=s_j | S_t=s_i)$
4. **Emission probabilities**: $B_{jk} = P(O_t=o_k | S_t=s_j)$
5. **Initial distribution**: $\pi_i = P(S_1=s_i)$

**Graphical model**:
```
S_1 → S_2 → S_3 → ... → S_T
 |     |     |           |
 v     v     v           v
O_1   O_2   O_3   ...   O_T
```

**Joint probability**:
$$P(S_{1:T}, O_{1:T}) = \pi_{S_1} \prod_{t=1}^{T-1} A_{S_t,S_{t+1}} \prod_{t=1}^{T} B_{S_t,O_t}$$

### 11.5.2 HMM Assumptions

**1. Markov assumption**: Future independent of past given present
$$P(S_{t+1}|S_{1:t}) = P(S_{t+1}|S_t)$$

**2. Output independence**: Observation depends only on current state
$$P(O_t|S_{1:t}, O_{1:t-1}) = P(O_t|S_t)$$

**3. Stationarity**: Transition/emission probabilities don't change over time

### 11.5.3 Three Fundamental Problems

**1. Evaluation**: Given model $\lambda=(A,B,\pi)$ and observations $O_{1:T}$, compute $P(O_{1:T}|\lambda)$
- **Solution**: Forward algorithm

**2. Decoding**: Find most likely state sequence
$$S^*_{1:T} = \arg\max_{S_{1:T}} P(S_{1:T}|O_{1:T}, \lambda)$$
- **Solution**: Viterbi algorithm

**3. Learning**: Estimate parameters $\lambda$ from observations
- **Solution**: Baum-Welch (EM) algorithm

## 11.6 Forward-Backward Algorithm

**Forward-Backward** computes state probabilities given observations.

### 11.6.1 Forward Algorithm

**Forward variable** $\alpha_t(i)$: Probability of observing $O_{1:t}$ and being in state $i$ at time $t$

$$\alpha_t(i) = P(O_1, ..., O_t, S_t=i | \lambda)$$

```
Algorithm: FORWARD(HMM λ, observations O_{1:T})

begin
    // Initialization
    for i = 1 to N do
α_1(i) ← π_i B_{i,O_1}
    
    // Recursion
    for t = 2 to T do
        for j = 1 to N do
α_t(j) ← [∑_{i=1}^{N} α_{t-1}(i) A_{ij}] B_{j,O_t}
    
    // Termination
    P(O_{1:T}) ← ∑_{i=1}^{N} α_T(i)
    
    return α, P(O_{1:T})
end
```

**Complexity**: $O(N^2 T)$

### 11.6.2 Backward Algorithm

**Backward variable** $\beta_t(i)$: Probability of future observations given current state

$$\beta_t(i) = P(O_{t+1}, ..., O_T | S_t=i, \lambda)$$

```
Algorithm: BACKWARD(HMM λ, observations O_{1:T})

begin
    // Initialization
    for i = 1 to N do
β_T(i) ← 1
    
    // Recursion (backward)
    for t = T-1 down to 1 do
        for i = 1 to N do
β_t(i) ← ∑_{j=1}^{N} A_{ij} B_{j,O_{t+1}} β_{t+1}(j)
    
    return β
end
```

### 11.6.3 State Probabilities

**Marginal state probability**: $P(S_t=i | O_{1:T})$

$$\gamma_t(i) = P(S_t=i | O_{1:T}) = \frac{\alpha_t(i)\beta_t(i)}{\sum_j \alpha_t(j)\beta_t(j)}$$

**Transition probability**: $P(S_t=i, S_{t+1}=j | O_{1:T})$

$$\xi_t(i,j) = \frac{\alpha_t(i) A_{ij} B_{j,O_{t+1}} \beta_{t+1}(j)}{P(O_{1:T})}$$

## 11.7 Viterbi Algorithm

**Viterbi** finds most likely state sequence (MAP inference).

### 11.7.1 Problem Formulation

**Goal**: Find
$$S^*_{1:T} = \arg\max_{S_{1:T}} P(S_{1:T} | O_{1:T})$$

**Key idea**: Dynamic programming

**Viterbi variable** $\delta_t(i)$: Probability of best path ending in state $i$ at time $t$

$$\delta_t(i) = \max_{S_{1:t-1}} P(S_{1:t-1}, S_t=i, O_{1:t})$$

### 11.7.2 Viterbi Algorithm

```
Algorithm: VITERBI(HMM λ, observations O_{1:T})

begin
    // Initialization
    for i = 1 to N do
δ_1(i) ← π_i B_{i,O_1}
ψ_1(i) ← 0
    
    // Recursion
    for t = 2 to T do
        for j = 1 to N do
            // Best previous state
δ_t(j) ← max_{i} [δ_{t-1}(i) A_{ij}] B_{j,O_t}
ψ_t(j) ← argmax_{i} [δ_{t-1}(i) A_{ij}]
    
    // Termination
    P* ← max_{i} δ_T(i)
    S*_T ← argmax_{i} δ_T(i)
    
    // Backtracking
    for t = T-1 down to 1 do
        S*_t ← ψ_{t+1}(S*_{t+1})
    
    return S*_{1:T}, P*
end
```

**Complexity**: $O(N^2 T)$

### 11.7.3 Comparison

| Algorithm | Computes | Output |
|-----------|----------|--------|
| Forward | $P(O_{1:T})$ | Likelihood |
| Forward-Backward | $P(S_t \mid O_{1:T})$ | State marginals |
| Viterbi | $\arg\max P(S_{1:T} \mid O_{1:T})$ | Best path |

## 11.8 Parameter Learning: Baum-Welch (EM)

**Baum-Welch** learns HMM parameters from unlabeled sequences.

### 11.8.1 Problem

**Given**: Observation sequences $O^{(1)}_{1:T}, ..., O^{(K)}_{1:T}$

**Find**: Parameters $\lambda = (A, B, \pi)$ that maximize likelihood
$$\lambda^* = \arg\max_\lambda \prod_{k=1}^{K} P(O^{(k)}_{1:T} | \lambda)$$

### 11.8.2 EM Algorithm

**E-step**: Compute expected sufficient statistics using Forward-Backward

**M-step**: Re-estimate parameters using expectations

```
Algorithm: BAUM-WELCH(sequences {O^{(k)}}, num_states N)

begin
    // Initialize parameters randomly
    A, B, π ← random normalized values
    
    repeat
        // E-step: Compute expectations for all sequences
        for each sequence k = 1 to K do
            α^{(k)}, β^{(k)} ← FORWARD-BACKWARD(A, B, π, O^{(k)})
            
            // Compute γ and ξ
            for t = 1 to T do
                for i = 1 to N do
γ^{(k)}_t(i) ← α^{(k)}_t(i) β^{(k)}_t(i) / P(O^{(k)})
                
                for j = 1 to N do
ξ^{(k)}_t(i,j) ← α^{(k)}_t(i) A_{ij} B_{j,O^{(k)}_{t+1}} β^{(k)}_{t+1}(j) / P(O^{(k)})
        
        // M-step: Update parameters
        // Initial probabilities
        for i = 1 to N do
π_i ← (∑_k γ^{(k)}_1(i)) / K
        
        // Transition probabilities
        for i = 1 to N do
            for j = 1 to N do
                A_{ij} ← (∑_k ∑_t ξ^{(k)}_t(i,j)) / (∑_k ∑_t γ^{(k)}_t(i))
        
        // Emission probabilities
        for j = 1 to N do
            for m = 1 to M do
                B_{j,m} ← (∑_k ∑_{t: O^{(k)}_t=m} γ^{(k)}_t(j)) / (∑_k ∑_t γ^{(k)}_t(j))
        
    until convergence
    
    return A, B, π
end
```

### 11.8.3 EM Intuition

**E-step**: "Fill in" missing states using current parameters
- Soft assignments via $\gamma_t(i)$ and $\xi_t(i,j)$

**M-step**: Update parameters using soft counts
- $A_{ij}$: Expected transitions $i \to j$ / Expected time in $i$
- $B_{jm}$: Expected emissions $m$ from $j$ / Expected time in $j$

**Convergence**: Guaranteed to increase likelihood (converges to local maximum)

## 11.9 Markov Random Fields

**Markov Random Field (MRF)** is an undirected graphical model.

### 11.9.1 Definition

**Structure**: Undirected graph $G = (V, E)$
- Nodes: Random variables
- Edges: Direct probabilistic interaction

**Joint distribution** factorizes over cliques:
$$P(X_1, ..., X_n) = \frac{1}{Z} \prod_{C \in \mathcal{C}} \psi_C(X_C)$$

where:
- $\mathcal{C}$: Set of maximal cliques
- $\psi_C$: Potential function (non-negative)
- $Z$: Partition function (normalization constant)
$$Z = \sum_{x_1,...,x_n} \prod_{C \in \mathcal{C}} \psi_C(x_C)$$

### 11.9.2 Markov Properties

**Pairwise Markov**: Non-adjacent variables independent given others
$$X_i \perp X_j | X_{V \setminus \{i,j\}} \text{ if } (i,j) \notin E$$

**Local Markov**: Variable independent of non-neighbors given neighbors
$$X_i \perp X_{V \setminus (\{i\} \cup N(i))} | X_{N(i)}$$

**Global Markov**: Sets separated by separator are independent

### 11.9.3 Log-Linear Models

**Parameterization**: Express potentials as exponentials
$$\psi_C(x_C) = \exp(\sum_k w_k f_k(x_C))$$

where $f_k$ are feature functions.

**Joint becomes**:
$$P(x) = \frac{1}{Z} \exp(\sum_C \sum_k w_k f_k(x_C))$$

### 11.9.4 Applications

**Image segmentation**:
- Nodes: Pixels
- Edges: Spatial neighbors
- Potentials: Favor similar labels for adjacent pixels

**Protein structure**:
- Nodes: Amino acids
- Edges: Spatial proximity
- Potentials: Interaction energies

## 11.10 Summary

### Graphical Models Comparison

| Property | Bayesian Network | Markov Network |
|----------|-----------------|----------------|
| Graph | Directed (DAG) | Undirected |
| Semantics | Conditional probabilities | Potentials |
| Factorization | $\prod P(X_i \mid \text{Pa}(X_i))$ | $\frac{1}{Z}\prod \psi_C(X_C)$ |
| Normalization | Automatic | Requires partition function |
| Causality | Natural | Not explicit |
| Loops | No (acyclic) | Yes |

### Inference Algorithms

| Algorithm | Graph Type | Exact? | Complexity |
|-----------|-----------|--------|------------|
| Variable Elimination | Any | Yes | Exp in treewidth |
| Belief Propagation | Tree | Yes | $O(n \cdot k^2)$ |
| Loopy BP | General | Approx | May not converge |
| Junction Tree | Any | Yes | Exp in treewidth |

### HMM Algorithms

| Algorithm | Task | Complexity |
|-----------|------|------------|
| Forward | Likelihood | $O(N^2T)$ |
| Forward-Backward | State marginals | $O(N^2T)$ |
| Viterbi | Best sequence | $O(N^2T)$ |
| Baum-Welch | Learning | $O(KN^2T)$ per iteration |

### Key Takeaways

**Graph structure**:
- Encodes independence assumptions
- Enables efficient inference
- Provides interpretability

**Inference**:
- Exact for trees (BP, VE)
- Approximate for general graphs
- Complexity depends on treewidth

**Learning**:
- EM for latent variables
- Maximum likelihood for parameters
- Structure learning is harder

**Applications**:
- Natural for many domains
- Combines prior knowledge with data
- Handles uncertainty explicitly

## 11.11 Implementation

For complete Python implementations, see:

[ch11_graphical_models_implementation.ipynb](ch11_graphical_models_implementation.ipynb)

The implementation notebook includes:

**Bayesian Networks**:
1. BN structure and CPD representation
2. Variable Elimination
3. D-separation checking
4. Inference queries

**Belief Propagation**:
1. Message passing on trees
2. Loopy BP for general graphs
3. Convergence monitoring

**Hidden Markov Models**:
1. Forward algorithm
2. Backward algorithm
3. Viterbi decoding
4. Baum-Welch learning
5. Speech recognition example
6. Part-of-speech tagging

**Markov Random Fields**:
1. MRF representation
2. Image denoising
3. Gibbs sampling

## Further Reading

### Textbooks

- **Koller, D., & Friedman, N. (2009).** *Probabilistic Graphical Models*. MIT Press. [THE definitive textbook]
- Aggarwal, C. C. (2021). *Artificial Intelligence: A Textbook*. Springer. [Chapter 11]
- Murphy, K. P. (2012). *Machine Learning: A Probabilistic Perspective*. MIT Press.
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.

### Classic Papers

**Bayesian Networks**:
- Pearl, J. (1988). *Probabilistic Reasoning in Intelligent Systems*. Morgan Kaufmann.
- Lauritzen, S. L., & Spiegelhalter, D. J. (1988). Local computations with probabilities. *Journal of the Royal Statistical Society*.

**HMMs**:
- Rabiner, L. R. (1989). A tutorial on hidden Markov models. *Proceedings of the IEEE*.
- Baum, L. E., et al. (1970). A maximization technique in statistical estimation of probabilistic functions. *Annals of Mathematical Statistics*.

**Belief Propagation**:
- Pearl, J. (1982). Reverend Bayes on inference engines. *AAAI*.
- Yedidia, J. S., Freeman, W. T., & Weiss, Y. (2003). Understanding belief propagation. *NIPS*.